# 交叉熵损失（Cross Entropy）手撕实现

## 1. 定义
对真实分布 $y$（one-hot，类别 $t$）与预测分布 $\hat p=\text{softmax}(z)$：
$$\mathcal L = -\sum_i y_i \log \hat p_i = -\log \hat p_t$$
batch 下取均值。

## 2. 数值稳定：log-softmax
朴素做法 `log(softmax(z))` 会先算 $\hat p$（已 safe）再取 log，但更稳的是直接用 **log-sum-exp**：
$$\log \hat p_i = z_i - m - \log\sum_j e^{z_j-m},\quad m=\max z$$
这样避免 `log(0)`（当 $\hat p_i\to 0$ 时），也少一次 exp/log 往返。PyTorch 的 `F.cross_entropy` 内部即 `F.nll_loss(F.log_softmax(z), t)`。

## 3. 梯度
$\frac{\partial \mathcal L}{\partial z_i}=\hat p_i - y_i$，即"预测概率减 one-hot"，非常简洁——这是 softmax+CE 组合的标准考点。

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
logits = torch.randn(3, 5, requires_grad=True)   # batch=3, vocab=5
labels = torch.tensor([2, 0, 4])

In [ ]:
# 朴素实现：softmax -> one-hot -> -sum(y*log(p))
def cross_entropy_naive(logits, labels):
    m = logits.max(dim=1, keepdim=True).values
    exp = torch.exp(logits - m)
    probs = exp / exp.sum(dim=1, keepdim=True)
    log_probs = torch.log(probs + 1e-12)
    n = logits.size(0)
    return -log_probs[torch.arange(n), labels].mean()

print('naive :', cross_entropy_naive(logits, labels).item())
print('torch :', F.cross_entropy(logits, labels).item())

In [ ]:
# 稳定实现：log-softmax（log-sum-exp），避免 log(0)
def cross_entropy_stable(logits, labels):
    m = logits.max(dim=1, keepdim=True).values
    shifted = logits - m
    logsumexp = shifted.logsumexp(dim=1)          # = log(sum(exp(shifted)))
    log_probs = shifted - logsumexp.unsqueeze(1)  # log_softmax
    n = logits.size(0)
    return -log_probs[torch.arange(n), labels].mean()

print('stable:', cross_entropy_stable(logits, labels).item())

In [ ]:
# 梯度验证：dL/dz = softmax(z) - one_hot
z = logits.detach().clone().requires_grad_(True)
loss = F.cross_entropy(z, labels)
loss.backward()
probs = F.softmax(z.detach(), dim=1)
onehot = F.one_hot(labels, num_classes=5).float()
print('autograd :', z.grad)
print('p - y    :', probs - onehot)
print('max diff :', (z.grad - (probs - onehot)).abs().max().item())

## 小结 / 易错点
- **不要** `log(softmax(z))`，要用 `log_softmax` / `log-sum-exp`，避免 `log(0)` 与精度损失。
- `F.cross_entropy` 的输入是 **logits**（未过 softmax），内部已含 log_softmax；若先 softmax 再 `F.nll_loss` 会损失精度且梯度错误。
- 梯度 $\hat p-y$ 形式极简，面试常考；多标签时改成 BCE-with-logits。
- batch 取**均值**还是求和要与下游学习率匹配，PyTorch 默认 `reduction='mean'`。

## ✅ 测试验证

In [ ]:
# 验证交叉熵实现与 PyTorch 内置一致
import torch
import torch.nn as nn

logits = torch.randn(4, 10)
labels = torch.randint(0, 10, (4,))

# PyTorch 基准
ref_loss = nn.CrossEntropyLoss()(logits, labels)

# 手动实现: -log(exp(x[label]) / sum(exp(x)))
log_softmax = logits.log_softmax(dim=-1)
manual_loss = -log_softmax.gather(1, labels.unsqueeze(1)).squeeze(1).mean()

assert torch.allclose(ref_loss, manual_loss, atol=1e-6), f"mismatch: {ref_loss} vs {manual_loss}"

# 数值稳定性: 大 logits 不溢出
big_logits = torch.tensor([[1000.0, 1001.0, 1002.0]])
big_labels = torch.tensor([2])
big_loss = nn.CrossEntropyLoss()(big_logits, big_labels)
assert not torch.isnan(big_loss), "overflow on large logits"

print(f"✅ CrossEntropy 测试通过: ref={ref_loss.item():.6f}, manual={manual_loss.item():.6f}")
